In [6]:
!pip install mne
import torch 
import kagglehub
import pandas as pd
import numpy as np
import mne  
import os
import glob
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, classification_report
import torch
from torch.utils.data import Dataset, DataLoader
from scipy.stats import zscore

In [ ]:
print("Verificando dataset...")
path = kagglehub.dataset_download("sigfest/database-for-emotion-recognition-system-gameemo")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Treinando usando: {device}")


In [ ]:
pasta_destino = "./data/processed/mne_raw"
os.makedirs(pasta_destino, exist_ok=True)
padrao_busca = os.path.join(path,"GAMEEMO","*","Raw EEG Data",".csv format","*.csv")
todos_arquivos = glob.glob(padrao_busca,recursive=True)
canais_oficiais = ['AF3', 'AF4', 'F3', 'F4', 'F7', 'F8', 'FC5', 'FC6', 'O1', 'O2', 'P7', 'P8', 'T7', 'T8'] #apenas os eletrodos do cerebro

for arquivo in todos_arquivos:
    nome_original = os.path.basename(arquivo)
    testado = nome_original[:3] 
    jogo = nome_original[3:5]

    print(f"Processando: Sujeito {testado} | Jogo {jogo}")

    df = pd.read_csv(arquivo)
    df_limpo = df[canais_oficiais] #retira colunas não relacionadas
    nomes_canais = list(df_limpo.columns)
    df_volts = df_limpo.T*1e-6 #converte de microvolts para volts
    taxa_amostragem=128 #taxa de frequencia utilizada pelo aparelho de EEG

    info = mne.create_info(ch_names=nomes_canais,sfreq=taxa_amostragem,ch_types='eeg') #explicita que os canais sao EEG, que a frequencia é 128 e os nomes das colunas
    info.set_montage('standard_1020') #mapeia no cérebro onde fica cada eletrodo
    raw = mne.io.RawArray(df_volts,info) #objeto que vai guardar a planilha do pandas e um dicionario com as informações pertinentes 

    nome_arquivo_saida = f"{testado}_{jogo}_raw.fif" 
    caminho_salvamento = os.path.join(pasta_destino, nome_arquivo_saida)
    raw.save(caminho_salvamento, overwrite=True)

    print(f"✅ Salvo com sucesso em: {caminho_salvamento}\n")
    print("\nObjeto MNE criado com sucesso!")
raw.plot_sensors(show_names=True)



In [ ]:
for arquivo in os.listdir(pasta_destino):
    print("1. Carregando o sinal bruto na RAM...")
    caminho_arquivo = os.path.join(pasta_destino,arquivo)
    raw = mne.io.read_raw_fif(caminho_arquivo, preload=True, verbose=False) #retira o arquivo do disco e coloca na RAM para processar

    print("2. Aplicando Filtro Passa-Banda (0.5 Hz - 45.0 Hz)...")
    raw.filter(l_freq=0.5,h_freq=45,fir_design='firwin',verbose=False) #filtra as frequencias com base nas frequencias cerebrais

    print("3. Cortando em Janelas de 1 segundo (com 50% de sobreposição)...")
    epochs = mne.make_fixed_length_epochs(raw,duration=1.0,overlap=0.5,preload=True,verbose=False) #cria janelas com sobreposicao de 1seg para resolver o problema de borda e aumenta o tamanho do dataset pra treinar o modelo
    matriz3d_numpy = epochs.get_data() #converte para uma matriz 3d (n_segundos, n_canais, nfotos_por_segundo)

    pasta_numpy = "./data/processed/numpy"
    os.makedirs(pasta_numpy,exist_ok=True)
    nome_numpy = os.path.basename(arquivo)
    testado,jogo,_ = nome_numpy.split("_")
    nome_arquivo = f"{testado}_{jogo}_tensor.npy"
    caminho_numpy = os.path.join(pasta_numpy,nome_arquivo)
    np.save(caminho_numpy,matriz3d_numpy)

    print(f"✅ Tensor 3D salvo com sucesso em: {caminho_numpy}")

In [ ]:
#filtra as frequencias com base nas frequencias cerebrais
def apply_frequency_filter(raw, l_freq=0.5, h_freq=45.0):   
    raw_filtrado = raw.copy()
    raw_filtrado.filter(l_freq=l_freq, h_freq=h_freq, fir_design='firwin', verbose=False)
    return raw_filtrado

def normalize_epochs(epochs_data):
    epochs_norm = zscore(epochs_data, axis=2) 
    epochs_norm = np.nan_to_num(epochs_norm) 
    return epochs_norm

def process_eeg_pipeline(pasta_origem, pasta_destino):
    os.makedirs(pasta_destino, exist_ok=True)
    arquivos = glob.glob(os.path.join(pasta_origem, "*.fif"))
    
    for caminho_arquivo in arquivos:
        nome_original = os.path.basename(caminho_arquivo)
        testado, jogo, _ = nome_original.split("_")
        
              # --- ETAPA DE ARTEFATOS (Justificativa) ---
        # Nota de Arquitetura: A remoção complexa de artefatos (ex: ICA para piscadas) 
        # foi omitida nesta iteração do baseline para focar na entrega end-to-end do pipeline. 
        # O filtro passa-banda a seguir atua como mitigador primário de ruído.
        raw = mne.io.read_raw_fif(caminho_arquivo, preload=True, verbose=False)
        raw_filtrado = apply_frequency_filter(raw)
        epochs = mne.make_fixed_length_epochs(raw_filtrado, duration=1.0, overlap=0.5, verbose=False)
        matriz_3d = epochs.get_data() #converte para uma matriz 3d (n_segundos, n_canais, nfotos_por_segundo)
        matriz_3d_normalizada = normalize_epochs(matriz_3d)
        
        nome_arquivo_saida = f"{testado}_{jogo}_tensor.npy"
        caminho_numpy = os.path.join(pasta_destino, nome_arquivo_saida)
        np.save(caminho_numpy, matriz_3d_normalizada)
        
    print(f"✅ Pipeline modular concluído! {len(arquivos)} arquivos processados e normalizados.")

In [ ]:
print("1. Iniciando Extração de Features (Feature Engineering)...")

X_list =[]
Y_list =[]
groups_list=[]
for arquivo in os.listdir(pasta_numpy):
    nome = os.path.basename(arquivo)
    caminho_arquivo = os.path.join(pasta_numpy,arquivo)
    sujeito,jogo,_ = nome.split("_")

    array = np.load(caminho_arquivo)
    media = np.mean(array,axis=2)
    desvio = np.std(array,axis=2)
    maximo = np.max(array,axis=2)
    minimo = np.min(array,axis=2)
    features = np.concatenate([media,desvio,maximo,minimo],axis=1)

    X_list.append(features)
    num_lotes = features.shape[0]
    Y_list.extend([jogo] * num_lotes)
    groups_list.extend([sujeito]*num_lotes)

X = np.vstack(X_list)
Y = np.array(Y_list)
groups = np.array(groups_list)

print(f"✅ Matriz 2D final criada! {X.shape[0]} amostras com {X.shape[1]} features.")
print("-" * 40)
print("2. Iniciando Treinamento e Validação K-Fold...")

logo = GroupKFold(n_splits=5)
rf = RandomForestClassifier(n_estimators=100,random_state=42,n_jobs=-1)
acuracias_por_sujeito = []
sujeito_atual = 1

for train_index, test_index in logo.split(X,Y,groups):
    X_test, Y_test = X[test_index], Y[test_index]
    X_train, Y_train = X[train_index],Y[train_index]

    sujeito_teste = groups[test_index][0]

    rf.fit(X_train,Y_train)
    chutado = rf.predict(X_test)
    acuracia = accuracy_score(Y_test,chutado)
    acuracias_por_sujeito.append(acuracia)

    print(f"Rodada {sujeito_atual:02d} | Testando no {sujeito_teste} | Acurácia: {acuracia:.2%}")
    sujeito_atual += 1
print(f"🎯 ACURÁCIA MÉDIA DO BASELINE: {np.mean(acuracias_por_sujeito):.2%}")

In [10]:
class EEGDataset(Dataset):
    def __init__(self,pasta_numpy):
        arquivos = glob.glob(os.path.join(pasta_numpy,"*.npy"))

        X_list =[]
        Y_list = []
        self.groups_list = []
        label_map = {"G1": 0, "G2": 1, "G3": 2, "G4": 3}
        print("1. Lendo os arquivos .npy do HD...")
        for arquivo in arquivos:
            nome = os.path.basename(arquivo)
            sujeito,jogo,_ = nome.split("_")
            tensor = np.load(arquivo)
            num_lotes = tensor.shape[0]
            X_list.append(tensor)
            Y_list.extend([label_map[jogo]] * num_lotes)
            self.groups_list.extend([sujeito]*num_lotes)

        X_numpy = np.vstack(X_list)
        Y_numpy = np.array(Y_list)
        self.groups = np.array(self.groups_list)

        print("2. Convertendo para Tensores do PyTorch (Alocando na RAM)...")
        self.X_tensor = torch.tensor(X_numpy, dtype=torch.float32)
        self.Y_tensor = torch.tensor(Y_numpy, dtype=torch.long)
        
        print(f"✅ Dataset pronto! Total de fatias de 1 segundo: {len(self.X_tensor)}")

    def __len__(self):
        return len(self.X_tensor)
    
    def __getitem__(self,idx):
        return self.X_tensor[idx], self.Y_tensor[idx]
        
print("-" * 40)
# Instancia o objeto apontando para a pasta que criamos na Fase 3
dataset_completo = EEGDataset("./data/processed/numpy")

# Simulando a Rede Neural pedindo a primeira amostra do dataset (índice 0)
sinal_teste, label_teste = dataset_completo[0]

print("-" * 40)
print(f"Formato de entrada (Sinal X): {sinal_teste.shape}") 
print(f"Classe de saída (Label y): {label_teste} (Tipo: {label_teste.dtype})")

----------------------------------------
1. Lendo os arquivos .npy do HD...
2. Convertendo para Tensores do PyTorch (Alocando na RAM)...
✅ Dataset pronto! Total de fatias de 1 segundo: 66752
----------------------------------------
Formato de entrada (Sinal X): torch.Size([14, 128])
Classe de saída (Label y): 3 (Tipo: torch.int64)
